# Phase 3 — Submission 00: Model 1 Deep LLaMA Causal Decoder (0-Context Ablation Baseline)

**Task:** Rhetorical Role Prediction in Legal Documents (Corporate Law — `CL` & Information Technology — `IT`)  
**Task Type:** Sentence-Level Multiclass Classification (13 Official Classes)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Target Backbone:** Phase 1 Model 1 Deep LLaMA Causal Decoder (134.7M, $L=16, d=768, H_q=12, H_{kv}=4$, GQA 3:1, SwiGLU $d_{ff}=2304$, RoPE)  
**Pretrained Hub Location:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `model1-deep-llama-135m`)  
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_3/sub00_model1_lorma_0context/`)  

---

### Architectural & Ablation Rationale
This notebook establishes the **0-Context Ablation Baseline** for Phase 3.
* **Isolated Sentence Input (0-Context):** Every sentence is fed in total isolation (`[target_tokens] + [<eos>]`) without any surrounding document context.
* **Strict Feature Parity with Sub 02:** Preserves identical 16-dim Fourier relative position encoding $\text{rel\_pos} = i / \max(1, N - 1)$, identical 8-dim learned domain embedding ($\text{CL}=0, \text{IT}=1$), identical 2-layer MLP classification head ($\text{Linear}(768 + 24 \to 256) \to \text{GELU} \to \text{Dropout}(0.1) \to \text{Linear}(256 \to 13)$), and identical True Multiplicative LORMA adaptation ($r=16, \alpha=32.0, \text{dropout}=0.05$).
* **Scientific Value:** Comparing Sub 00 (0-Context) against Sub 02 (Full 512-Token Packed Context with 128-token overlap) directly quantifies the exact performance gain of inter-sentence discourse context in legal reasoning.


In [ ]:
# Cell 2: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys
import json
import math
import time
import zipfile
import random
import getpass
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import PreTrainedTokenizerFast, AutoConfig, AutoModel, get_cosine_schedule_with_warmup
from huggingface_hub import hf_hub_download, HfApi

# 1. Reproducible Seed Initialization
def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Compute Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

# 3. Repository and Target Configuration
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_3/sub00_model1_lorma_0context"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# 4. Upfront Write-Permission Token Check (Fails Fast within 5 seconds if invalid)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF") or os.environ.get("HUGGINGFACE_TOKEN") or ""
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HF")
    except Exception:
        pass

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass
    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)
        print(f"Successfully verified write permissions on Hugging Face Hub: '{MODEL_REPO}'")
    except Exception as e:
        print(f"Warning: Upfront HF token verification encountered: {e}")
else:
    print("Notice: HF_TOKEN not found in environment. You can enter it before upload.")


In [ ]:
# Cell 3: Universal Dataset Loader for CL & IT Splits
LABEL2ID = {
    "Fact": 0, "Issue": 1, "ArgumentPetitioner": 2, "ArgumentRespondent": 3,
    "Statute": 4, "Dissent": 5, "PrecedentReliedUpon": 6, "PrecedentNotReliedUpon": 7,
    "PrecedentOverruled": 8, "RulingByLowerCourt": 9, "RatioOfTheDecision": 10,
    "RulingByPresentCourt": 11, "None": 12
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 13
DOMAIN2ID = {"CL": 0, "IT": 1}
ID2DOMAIN = {0: "CL", 1: "IT"}

@dataclass
class LegalDocument:
    doc_id: str
    domain: str
    domain_id: int
    sentences: List[str]
    labels: Optional[List[str]] = None
    label_ids: Optional[List[int]] = None

def normalize_dataset_records(raw_data: Any, domain: str, label2id: Dict[str, int] = LABEL2ID) -> List[LegalDocument]:
    domain_id = DOMAIN2ID.get(domain.upper(), 0)
    normalized: List[LegalDocument] = []
    if isinstance(raw_data, list):
        for item in raw_data:
            if not isinstance(item, dict):
                continue
            doc_id = str(item.get("id") or item.get("doc_id") or item.get("name") or "")
            sentences = item.get("text") or item.get("sentences") or []
            labels = item.get("labels") or item.get("label") or None
            label_ids = None
            if labels is not None:
                assert len(sentences) == len(labels), f"Length mismatch in doc {doc_id}"
                label_ids = []
                for lbl in labels:
                    if lbl not in label2id:
                        raise ValueError(f"CRITICAL: Unknown label '{lbl}' in doc '{doc_id}'!")
                    label_ids.append(label2id[lbl])
            normalized.append(LegalDocument(doc_id=doc_id, domain=domain.upper(), domain_id=domain_id, sentences=sentences, labels=labels, label_ids=label_ids))
    elif isinstance(raw_data, dict):
        for doc_id, val in raw_data.items():
            if isinstance(val, list):
                normalized.append(LegalDocument(doc_id=str(doc_id), domain=domain.upper(), domain_id=domain_id, sentences=val, labels=None, label_ids=None))
            elif isinstance(val, dict):
                sentences = val.get("text") or val.get("sentences") or []
                labels = val.get("labels") or val.get("label") or None
                label_ids = None
                if labels is not None:
                    label_ids = []
                    for lbl in labels:
                        if lbl not in label2id:
                            raise ValueError(f"CRITICAL: Unknown label '{lbl}' in doc '{doc_id}'!")
                        label_ids.append(label2id[lbl])
                normalized.append(LegalDocument(doc_id=str(doc_id), domain=domain.upper(), domain_id=domain_id, sentences=sentences, labels=labels, label_ids=label_ids))
    return normalized

def load_all_datasets(repo_id: str = DATASET_REPO):
    print("Downloading Phase 3 datasets from Hugging Face Hub...")
    vocab_path = hf_hub_download(repo_id, "Phase_3_RR/Train/label_vocab.json", repo_type="dataset", token=HF_TOKEN or None)
    with open(vocab_path, "r", encoding="utf-8") as f:
        raw_vocab = json.load(f)
    if isinstance(raw_vocab, list):
        vocab = {lbl: i for i, lbl in enumerate(raw_vocab)}
    else:
        vocab = raw_vocab
    assert vocab == LABEL2ID, f"Vocabulary parity mismatch! Hub: {vocab} vs Code: {LABEL2ID}"
        
    splits = {
        "cl_train": ("Phase_3_RR/Train/CL_train.json", "CL"),
        "it_train": ("Phase_3_RR/Train/IT_train.json", "IT"),
        "cl_dev":   ("Phase_3_RR/Train/CL_dev.json", "CL"),
        "it_dev":   ("Phase_3_RR/Train/IT_dev.json", "IT"),
        "cl_test":  ("Phase_3_RR/Test/CL_test.json", "CL"),
        "it_test":  ("Phase_3_RR/Test/IT_test.json", "IT"),
    }
    raw = {}
    for key, (path, _) in splits.items():
        loc = hf_hub_download(repo_id, path, repo_type="dataset", token=HF_TOKEN or None)
        with open(loc, "r", encoding="utf-8") as f:
            raw[key] = json.load(f)
            
    train_docs = normalize_dataset_records(raw["cl_train"], "CL") + normalize_dataset_records(raw["it_train"], "IT")
    dev_docs = normalize_dataset_records(raw["cl_dev"], "CL") + normalize_dataset_records(raw["it_dev"], "IT")
    test_docs = normalize_dataset_records(raw["cl_test"], "CL") + normalize_dataset_records(raw["it_test"], "IT")
    assert not ({d.doc_id for d in train_docs if d.domain == "CL"} & {d.doc_id for d in train_docs if d.domain == "IT"}), "Train doc ID collision!"
    assert not ({d.doc_id for d in dev_docs if d.domain == "CL"} & {d.doc_id for d in dev_docs if d.domain == "IT"}), "Dev doc ID collision!"
    assert not ({d.doc_id for d in test_docs if d.domain == "CL"} & {d.doc_id for d in test_docs if d.domain == "IT"}), "Test doc ID collision!"
    return train_docs, dev_docs, test_docs, vocab

train_docs, dev_docs, test_docs, label_vocab = load_all_datasets()
print(f"Loaded: {len(train_docs)} Train Docs (80 expected), {len(dev_docs)} Dev Docs (10 expected), {len(test_docs)} Test Docs (10 expected)")


In [ ]:
# Cell 4: Tokenizer & 0-Context Sentence Dataset Assembly
def load_model1_tokenizer(repo_id: str = MODEL_REPO):
    tok_path = hf_hub_download(repo_id, "model1-deep-llama-135m/tokenizer.json", token=HF_TOKEN or None)
    tok = PreTrainedTokenizerFast(
        tokenizer_file=tok_path,
        bos_token="<bos>",
        eos_token="<eos>",
        pad_token="<pad>",
        unk_token="<unk>"
    )
    tok.padding_side = "right"
    assert tok.pad_token_id == 3 and tok.eos_token_id == 1, "Special token ID mismatch!"
    return tok

tokenizer = load_model1_tokenizer()

@dataclass
class ZeroContextSentenceSample:
    doc_id: str
    sent_idx: int
    text: str
    input_ids: List[int]
    rel_pos: float
    domain_id: int
    label_id: int  # -100 if unknown

def extract_0context_samples(docs: List[LegalDocument], tokenizer: PreTrainedTokenizerFast, max_sent_len: int = 128) -> List[ZeroContextSentenceSample]:
    samples: List[ZeroContextSentenceSample] = []
    for doc in docs:
        N = len(doc.sentences)
        for i, sent in enumerate(doc.sentences):
            # 0-Context: Target sentence alone, truncated to max_sent_len - 1, followed strictly by native EOS
            toks = tokenizer.encode(sent, add_special_tokens=False)[: max_sent_len - 1]
            toks.append(tokenizer.eos_token_id)
            rel_pos = float(i) / float(max(1, N - 1))
            lbl_id = doc.label_ids[i] if doc.label_ids is not None else -100
            samples.append(ZeroContextSentenceSample(
                doc_id=doc.doc_id,
                sent_idx=i,
                text=sent,
                input_ids=toks,
                rel_pos=rel_pos,
                domain_id=doc.domain_id,
                label_id=lbl_id
            ))
    return samples

train_samples = extract_0context_samples(train_docs, tokenizer)
dev_samples = extract_0context_samples(dev_docs, tokenizer)
test_samples = extract_0context_samples(test_docs, tokenizer)

print(f"Extracted 0-Context Samples -> Train: {len(train_samples):,}, Dev: {len(dev_samples):,}, Test: {len(test_samples):,}")

class ZeroContextRRDataset(Dataset):
    def __init__(self, samples: List[ZeroContextSentenceSample]):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

def collate_0context_batch(batch: List[ZeroContextSentenceSample], pad_token_id: int = 3):
    batch_size = len(batch)
    lengths = [len(s.input_ids) for s in batch]
    max_len = max(lengths)
    
    padded_input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long)
    padded_attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    eos_positions = torch.zeros(batch_size, dtype=torch.long)
    rel_positions = torch.zeros(batch_size, dtype=torch.float32)
    domain_ids = torch.zeros(batch_size, dtype=torch.long)
    label_ids = torch.zeros(batch_size, dtype=torch.long)
    
    doc_ids = []
    sent_indices = []
    
    for i, s in enumerate(batch):
        seq_len = len(s.input_ids)
        padded_input_ids[i, :seq_len] = torch.tensor(s.input_ids, dtype=torch.long)
        padded_attention_mask[i, :seq_len] = 1
        eos_positions[i] = seq_len - 1  # In isolated sentence, EOS is at the final position
        rel_positions[i] = s.rel_pos
        domain_ids[i] = s.domain_id
        label_ids[i] = s.label_id
        doc_ids.append(s.doc_id)
        sent_indices.append(s.sent_idx)
        
    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "eos_pos": eos_positions,
        "rel_pos": rel_positions,
        "domain_ids": domain_ids,
        "label_ids": label_ids,
        "doc_ids": doc_ids,
        "sent_indices": sent_indices
    }


In [ ]:
# Cell 5: Fourier Relative Position Encodings & 2-Layer MLP Classification Head
class FourierPositionEncoding(nn.Module):
    """
    16-dimensional Fourier relative position encoding:
      rel_pos = i / max(1, N - 1) in [0.0, 1.0].
    Computes 8 frequencies: omega_k = pi * 2^k for k in 0..7.
    Outputs: [sin(rel_pos * omega), cos(rel_pos * omega)] -> 16 dimensions.
    """
    def __init__(self, dim: int = 16):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim
        self.num_bands = dim // 2
        freq_bands = (2.0 ** torch.arange(self.num_bands).float()) * math.pi
        self.register_buffer("freq_bands", freq_bands)
        
    def forward(self, rel_pos: torch.Tensor) -> torch.Tensor:
        scaled = rel_pos.unsqueeze(-1) * self.freq_bands.to(rel_pos.device, dtype=rel_pos.dtype)
        return torch.cat([torch.sin(scaled), torch.cos(scaled)], dim=-1)

class RhetoricalRoleMLPHead(nn.Module):
    """
    2-Layer MLP Classification Head (Identical across Sub 00 and Sub 02):
      LayerNorm(d_model) -> Concat [LayerNorm(h_snt), 16-dim Fourier, 8-dim Domain]
      Linear(768 + 24 -> 256) -> GELU -> Dropout(0.1) -> Linear(256 -> 13).
    """
    def __init__(
        self,
        d_model: int = 768,
        num_classes: int = 13,
        fourier_dim: int = 16,
        domain_dim: int = 8,
        hidden_dim: int = 256,
        dropout: float = 0.1
    ):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.fourier_pos = FourierPositionEncoding(dim=fourier_dim)
        self.domain_embed = nn.Embedding(num_embeddings=2, embedding_dim=domain_dim)
        
        in_features = d_model + fourier_dim + domain_dim  # 768 + 16 + 8 = 792
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
        nn.init.normal_(self.domain_embed.weight, mean=0.0, std=0.02)
        
    def forward(self, sentence_reprs: torch.Tensor, rel_pos: torch.Tensor, domain_ids: torch.Tensor) -> torch.Tensor:
        h_norm = self.layer_norm(sentence_reprs)
        fourier_feats = self.fourier_pos(rel_pos)
        domain_feats = self.domain_embed(domain_ids)
        combined = torch.cat([h_norm, fourier_feats, domain_feats], dim=-1)
        hidden = self.dropout(self.act(self.fc1(combined)))
        return self.fc2(hidden)


In [ ]:
# Cell 6: True Multiplicative LORMA (Low-Rank Multiplicative Adaptation)
class TrueLORMALinear(nn.Module):
    """
    True Multiplicative LORMA Layer:
      W = (I + (alpha / r) * B @ A) * W_0
      y = x @ W^T = x @ W_0^T + (alpha / r) * (x @ W_0^T) @ A^T @ B^T
      h0 = base_linear(x)
      y = h0 + (alpha / r) * lorma_B(lorma_A(dropout(h0)))
    """
    def __init__(self, base_linear: nn.Linear, r: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        
        # Frozen base weight
        self.base_linear = base_linear
        for p in self.base_linear.parameters():
            p.requires_grad = False
            
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Trainable low-rank multiplicative adapter matrices
        self.lorma_A = nn.Linear(self.out_features, r, bias=False, device=base_linear.weight.device, dtype=base_linear.weight.dtype)
        self.lorma_B = nn.Linear(r, self.out_features, bias=False, device=base_linear.weight.device, dtype=base_linear.weight.dtype)
        
        # Kaiming uniform on A, zeros on B (guarantees step 0 identity)
        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.base_linear(x)
        delta_h = self.lorma_B(self.lorma_A(self.dropout(h0)))
        return h0 + self.scaling * delta_h

def apply_lorma_to_model(
    model: nn.Module,
    r: int = 16,
    alpha: float = 32.0,
    dropout: float = 0.05,
    target_modules: Tuple[str, ...] = ("q_proj", "k_proj", "v_proj", "o_proj")
) -> int:
    # 1. Freeze all base backbone parameters
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. Dynamically replace target attention projections with TrueLORMALinear
    replaced_count = 0
    for name, module in model.named_modules():
        for target in target_modules:
            if hasattr(module, target):
                orig_layer = getattr(module, target)
                if isinstance(orig_layer, nn.Linear):
                    lorma_layer = TrueLORMALinear(orig_layer, r=r, alpha=alpha, dropout=dropout)
                    setattr(module, target, lorma_layer)
                    replaced_count += 1
    return replaced_count


In [ ]:
# Cell 7: Model 1 Backbone Loading & LORMA Injection
from safetensors.torch import load_file as safetensors_load_file

print("Downloading Model 1 Deep LLaMA configuration and weights from Hugging Face Hub...")
cfg_path = hf_hub_download(MODEL_REPO, "model1-deep-llama-135m/config.json")
weights_path = hf_hub_download(MODEL_REPO, "model1-deep-llama-135m/model.safetensors")

config = AutoConfig.from_pretrained(cfg_path)
config.use_cache = False
base_model = AutoModel.from_config(config)

raw_state_dict = safetensors_load_file(weights_path)
cleaned_sd = {}
for k, v in raw_state_dict.items():
    if k.startswith("model."):
        cleaned_sd[k[len("model."):]] = v
    elif not k.startswith("lm_head."):
        cleaned_sd[k] = v

load_res = base_model.load_state_dict(cleaned_sd, strict=True)
assert len(load_res.missing_keys) == 0, f"Missing keys when loading backbone: {load_res.missing_keys}"
assert len(load_res.unexpected_keys) == 0, f"Unexpected keys when loading backbone: {load_res.unexpected_keys}"
print(f"Base Model 1 loaded successfully with strict=True! 100% of Phase 1 weights matched ({len(cleaned_sd)} tensors).")

# Inject True Multiplicative LORMA
replaced_modules = apply_lorma_to_model(base_model, r=16, alpha=32.0, dropout=0.05)
print(f"Applied True Multiplicative LORMA to {replaced_modules} projection layers (q, k, v, o).")

# Instantiate MLP Classification Head
head = RhetoricalRoleMLPHead(d_model=768, num_classes=13, dropout=0.1)

# CRITICAL: Move entire model and head to device post-injection to synchronize parameter devices
base_model = base_model.to(device)
head = head.to(device)

# Audit Parameter Counts
trainable_lorma = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
trainable_head = sum(p.numel() for p in head.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in base_model.parameters()) + sum(p.numel() for p in head.parameters())
trainable_total = trainable_lorma + trainable_head

print("\n" + "="*50)
print(f"Backbone Total Parameters:  {sum(p.numel() for p in base_model.parameters()):,}")
print(f"LORMA Trainable Parameters:  {trainable_lorma:,} ({trainable_lorma / total_params * 100:.3f}%)")
print(f"MLP Head Trainable Params:   {trainable_head:,} ({trainable_head / total_params * 100:.3f}%)")
print(f"Total Trainable Parameters:  {trainable_total:,} ({trainable_total / total_params * 100:.3f}%)")
print("="*50 + "\n")


In [ ]:
# Cell 8: Smoothed Inverse Square-Root Class Weighting
def compute_smoothed_weights(train_samples: List[ZeroContextSentenceSample], num_classes: int = 13, clamp_min: float = 0.35, clamp_max: float = 6.0):
    counts = [0] * num_classes
    for s in train_samples:
        if 0 <= s.label_id < num_classes:
            counts[s.label_id] += 1
    total = sum(counts)
    weights = []
    for c in range(num_classes):
        raw_w = math.sqrt(total / (num_classes * max(1, counts[c])))
        clamped_w = max(clamp_min, min(clamp_max, raw_w))
        weights.append(clamped_w)
    return torch.tensor(weights, dtype=torch.float32)

class_weights = compute_smoothed_weights(train_samples).to(device)
print(f"Smoothed Class Weights computed across {len(train_samples):,} training sentences.")
for lbl, idx in sorted(LABEL2ID.items(), key=lambda x: x[1]):
    print(f"  [{idx:2d}] {lbl:<24}: {class_weights[idx].item():.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
# Cell 9: Differential Optimization & Training Loop
BATCH_SIZE = 64
EPOCHS = 5
GRAD_ACCUM_STEPS = 5
LR_BACKBONE = 3.0e-4
LR_HEAD = 1.0e-3

train_loader = DataLoader(
    ZeroContextRRDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_0context_batch,
    drop_last=False
)

dev_loader = DataLoader(
    ZeroContextRRDataset(dev_samples),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collate_0context_batch
)

# Differential Optimizer Parameter Groups
param_groups = [
    {
        "params": [p for p in base_model.parameters() if p.requires_grad],
        "lr": LR_BACKBONE,
        "weight_decay": 0.01,
    },
    {
        "params": [p for p in head.parameters() if p.requires_grad],
        "lr": LR_HEAD,
        "weight_decay": 0.01,
    },
]

optimizer = torch.optim.AdamW(param_groups)
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f"Optimizer: AdamW | Total Training Steps: {total_steps} | Warmup Steps: {warmup_steps}")

# Macro-F1 Metric Evaluation
def compute_macro_f1(gold_labels: List[int], pred_labels: List[int], num_classes: int = 13):
    tp = [0] * num_classes
    fp = [0] * num_classes
    fn = [0] * num_classes
    for g, p in zip(gold_labels, pred_labels):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    f1s = []
    for c in range(num_classes):
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / num_classes

def evaluate(model, head, data_loader, criterion, device):
    model.eval()
    head.eval()
    total_loss = 0.0
    all_preds = []
    all_golds = []
    all_logits = []
    all_doc_ids = []
    all_sent_indices = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            eos_pos = batch["eos_pos"].to(device)
            rel_pos = batch["rel_pos"].to(device)
            domain_ids = batch["domain_ids"].to(device)
            label_ids = batch["label_ids"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_indices = torch.arange(input_ids.size(0), device=device)
            sent_reprs = outputs.last_hidden_state[batch_indices, eos_pos]
            
            logits = head(sent_reprs, rel_pos, domain_ids)
            loss = criterion(logits, label_ids)
            total_loss += loss.item() * input_ids.size(0)
            
            preds = logits.argmax(dim=-1).cpu().tolist()
            golds = label_ids.cpu().tolist()
            all_preds.extend(preds)
            all_golds.extend(golds)
            all_logits.extend(logits.cpu().tolist())
            all_doc_ids.extend(batch["doc_ids"])
            all_sent_indices.extend(batch["sent_indices"])
            
    avg_loss = total_loss / len(data_loader.dataset)
    macro_f1 = compute_macro_f1(all_golds, all_preds)
    dev_logits_map = {}
    for d_id, s_idx, l_row in zip(all_doc_ids, all_sent_indices, all_logits):
        if d_id not in dev_logits_map:
            dev_logits_map[d_id] = {}
        dev_logits_map[d_id][s_idx] = l_row
    return avg_loss, macro_f1, all_golds, all_preds, dev_logits_map

# 5-Epoch Training Loop
os.makedirs("checkpoint_sub00", exist_ok=True)
best_dev_f1 = 0.0
history = {"train_loss": [], "dev_loss": [], "dev_f1": []}

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

print("Commencing 5-epoch training loop for Sub 00 (0-Context LORMA Model 1)...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    base_model.train()
    head.train()
    running_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        eos_pos = batch["eos_pos"].to(device)
        rel_pos = batch["rel_pos"].to(device)
        domain_ids = batch["domain_ids"].to(device)
        label_ids = batch["label_ids"].to(device)
        
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
            batch_indices = torch.arange(input_ids.size(0), device=device)
            sent_reprs = outputs.last_hidden_state[batch_indices, eos_pos]
            
            logits = head(sent_reprs, rel_pos, domain_ids)
            loss = criterion(logits, label_ids)
            loss = loss / GRAD_ACCUM_STEPS
            
        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS
        
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
    epoch_train_loss = running_loss / len(train_loader)
    dev_loss, dev_f1, _, _, _ = evaluate(base_model, head, dev_loader, criterion, device)
    
    history["train_loss"].append(epoch_train_loss)
    history["dev_loss"].append(dev_loss)
    history["dev_f1"].append(dev_f1)
    
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} | Dev Loss: {dev_loss:.4f} | Dev Macro-F1: {dev_f1:.4f}")
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        torch.save(base_model.state_dict(), "checkpoint_sub00/best_model_weights.pt")
        torch.save(head.state_dict(), "checkpoint_sub00/best_head_weights.pt")
        print(f" -> New Best Dev Macro-F1: {best_dev_f1:.4f}! Checkpoint saved.")

total_train_time = time.time() - start_time
print(f"\nTraining completed in {total_train_time / 60:.2f} minutes. Best Dev Macro-F1: {best_dev_f1:.4f}")


In [ ]:
# Cell 10: Final Evaluation, Detailed Classification Report & Visualization
import matplotlib.pyplot as plt

# Load best checkpoint
base_model.load_state_dict(torch.load("checkpoint_sub00/best_model_weights.pt", map_location=device))
head.load_state_dict(torch.load("checkpoint_sub00/best_head_weights.pt", map_location=device))
print("Loaded best weights for final evaluation.")

final_dev_loss, final_dev_f1, all_golds, all_preds, dev_logits_map = evaluate(base_model, head, dev_loader, criterion, device)

# Export ordered dev_logits.pt
ordered_dev_logits = {}
for doc in dev_docs:
    d_map = dev_logits_map[doc.doc_id]
    ordered_dev_logits[doc.doc_id] = torch.tensor([d_map[i] for i in range(len(doc.sentences))], dtype=torch.float32)
torch.save(ordered_dev_logits, "checkpoint_sub00/dev_logits.pt")
print(f"Exported dev soft logits for {len(ordered_dev_logits)} documents to checkpoint_sub00/dev_logits.pt")

# Per-Class Performance Summary
tp = [0] * NUM_CLASSES
fp = [0] * NUM_CLASSES
fn = [0] * NUM_CLASSES
for g, p in zip(all_golds, all_preds):
    if g == p:
        tp[g] += 1
    else:
        fp[p] += 1
        fn[g] += 1

per_class_summary = {}
print("\n" + "="*70)
print(f"{'Class Name':<25} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<8}")
print("="*70)
for c in range(NUM_CLASSES):
    c_name = ID2LABEL[c]
    p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
    r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    support = tp[c] + fn[c]
    per_class_summary[c_name] = {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4), "support": support}
    print(f"{c_name:<25} {p:<10.4f} {r:<10.4f} {f1:<10.4f} {support:<8d}")
print("="*70)
print(f"{'Overall Macro-F1':<25} {'':<10} {'':<10} {final_dev_f1:<10.4f} {len(all_golds):<8d}")
print("="*70 + "\n")

# Save Metrics JSON
metrics_report = {
    "model_id": "model1-deep-llama-135m",
    "adaptation": "True Multiplicative LORMA (0-Context Ablation Baseline)",
    "best_dev_macro_f1": round(final_dev_f1, 4),
    "dev_loss": round(final_dev_loss, 4),
    "total_params": total_params,
    "trainable_params": trainable_total,
    "trainable_percentage": round((trainable_total / total_params) * 100, 4),
    "per_class": per_class_summary,
    "training_time_minutes": round(total_train_time / 60, 2),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD
}

with open("checkpoint_sub00/val_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_report, f, indent=2)

with open("checkpoint_sub00/training_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

# Plot Loss & Macro-F1 Progression
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o")
plt.plot(range(1, EPOCHS + 1), history["dev_loss"], label="Dev Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sub 00: Cross-Entropy Loss Curve")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history["dev_f1"], label="Dev Macro-F1", marker="^", color="green")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Sub 00: Dev Macro-F1 Progression")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("checkpoint_sub00/loss_plot.png", dpi=150)
plt.show()
print("Saved evaluation report and loss plot to 'checkpoint_sub00/loss_plot.png'.")


In [ ]:
# Cell 11: Test Inference & Codabench Packaging
test_loader = DataLoader(
    ZeroContextRRDataset(test_samples),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collate_0context_batch
)

base_model.eval()
head.eval()

test_preds = []
test_doc_ids = []
test_sent_indices = []
test_logits = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        eos_pos = batch["eos_pos"].to(device)
        rel_pos = batch["rel_pos"].to(device)
        domain_ids = batch["domain_ids"].to(device)
        
        outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
        batch_indices = torch.arange(input_ids.size(0), device=device)
        sent_reprs = outputs.last_hidden_state[batch_indices, eos_pos]
        
        logits = head(sent_reprs, rel_pos, domain_ids)
        preds = logits.argmax(dim=-1).cpu().tolist()
        
        test_preds.extend(preds)
        test_doc_ids.extend(batch["doc_ids"])
        test_sent_indices.extend(batch["sent_indices"])
        test_logits.extend(logits.cpu().tolist())

# Reconstruct document-level predictions
predictions_dict: Dict[str, List[str]] = {}
for doc_id, s_idx, pred_id in zip(test_doc_ids, test_sent_indices, test_preds):
    if doc_id not in predictions_dict:
        target_doc = next(d for d in test_docs if d.doc_id == doc_id)
        predictions_dict[doc_id] = [None] * len(target_doc.sentences)
    predictions_dict[doc_id][s_idx] = ID2LABEL[pred_id]

# Verification Assertions
assert len(predictions_dict) == len(test_docs), f"Expected {len(test_docs)} docs, got {len(predictions_dict)}"
for doc in test_docs:
    assert doc.doc_id in predictions_dict, f"Missing doc: {doc.doc_id}"
    doc_preds = predictions_dict[doc.doc_id]
    assert len(doc_preds) == len(doc.sentences), f"Sentence count mismatch for doc {doc.doc_id}"
    assert None not in doc_preds, f"Unpredicted sentence in doc {doc.doc_id}"
    for lbl in doc_preds:
        assert lbl in LABEL2ID, f"Invalid label '{lbl}'"

print(f"Successfully generated test predictions for {len(predictions_dict)} documents ({sum(len(v) for v in predictions_dict.values()):,} sentences).")

# Save test_logits.pt
test_logits_map = {}
for d_id, s_idx, l_row in zip(test_doc_ids, test_sent_indices, test_logits):
    if d_id not in test_logits_map:
        test_logits_map[d_id] = {}
    test_logits_map[d_id][s_idx] = l_row

ordered_test_logits = {}
for doc in test_docs:
    d_map = test_logits_map[doc.doc_id]
    ordered_test_logits[doc.doc_id] = torch.tensor([d_map[i] for i in range(len(doc.sentences))], dtype=torch.float32)
torch.save(ordered_test_logits, "checkpoint_sub00/test_logits.pt")
print(f"Exported test soft logits to checkpoint_sub00/test_logits.pt ({os.path.getsize('checkpoint_sub00/test_logits.pt'):,} bytes).")

# Save prediction.json
pred_path = "checkpoint_sub00/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions_dict, f, indent=2)

# Package into submission.zip with prediction.json strictly at root
zip_path = "checkpoint_sub00/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

# Verify submission.zip root integrity
with zipfile.ZipFile(zip_path, "r") as zf:
    infolist = zf.infolist()
    assert len(infolist) == 1, f"Expected exactly 1 file in submission.zip, found {len(infolist)}"
    assert infolist[0].filename == "prediction.json", f"Found {infolist[0].filename} instead of prediction.json"

print(f"Verified submission.zip: {os.path.getsize(zip_path):,} bytes, root file 'prediction.json'.")


In [ ]:
# Cell 12: Local Artifact Preparation & Atomic Hugging Face Hub Upload
output_dir = "checkpoint_sub00"

# 1. Save config and tokenizer files
tokenizer.save_pretrained(output_dir)
config.save_pretrained(output_dir)

# 2. Extract and save LORMA weights separately
lorma_state = {k: v for k, v in base_model.state_dict().items() if "lorma_" in k}
torch.save(lorma_state, os.path.join(output_dir, "model1_lorma_weights.pt"))
print(f"Saved LORMA adapter weights: {len(lorma_state)} tensors.")

# 3. Save MLP Head weights
torch.save(head.state_dict(), os.path.join(output_dir, "mlp_head.pt"))
print("Saved RhetoricalRoleMLPHead weights.")

# 4. Verify all required artifacts exist locally
required_files = [
    "model1_lorma_weights.pt",
    "mlp_head.pt",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "val_metrics.json",
    "training_evaluation_report.json",
    "loss_plot.png",
    "dev_logits.pt",
    "test_logits.pt",
    "prediction.json",
    "submission.zip"
]

missing_files = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
assert not missing_files, f"Cannot proceed with upload! Missing local files: {missing_files}"
print(f"All {len(required_files)} local artifacts verified present and ready for upload!")

# 5. Atomic single-commit folder upload to Hugging Face Hub
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")

# Write compliant README.md for Hugging Face Hub model card
readme_path = os.path.join(output_dir, "README.md")
readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- rhetorical-roles
---

# Phase 3 Sub 00: Model 1 0-Context LORMA Ablation Baseline
- **Base Model:** `{MODEL_REPO}`
- **Task:** Sentence-Level Rhetorical Role Prediction (13 Classes)
- **Dev Macro-F1:** {final_dev_f1:.4f}
"""
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

api = HfApi(token=HF_TOKEN)
print(f"Initiating atomic upload to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 3 Sub 00 (0-Context LORMA Model 1 Ablation) - Dev F1: {final_dev_f1:.4f}",
    token=HF_TOKEN
)
print(f"\nUpload complete! Artifacts published to:\nhttps://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")
